## 1. Setup and Load Cleaned Data

In [15]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [16]:
# Load cleaned data from Part 1.1
data_path = '../data/processed/'

daily_revenue = pd.read_csv(data_path + 'daily_revenue.csv', parse_dates=['Date'])
sale_ticket_clean = pd.read_csv(data_path + 'sale_ticket_clean.csv', parse_dates=['Date'])
customer_analysis = pd.read_csv(data_path + 'customer_analysis.csv', parse_dates=['First_Visit', 'Last_Visit'])
employee_performance = pd.read_csv(data_path + 'employee_performance.csv')
rpt_by_service = pd.read_csv(data_path + 'rpt_by_service.csv')

print("✓ All cleaned datasets loaded!")
print(f"\nDataset Shapes:")
print(f"Daily Revenue: {daily_revenue.shape}")
print(f"Sale Ticket Detail: {sale_ticket_clean.shape}")
print(f"Customer Analysis: {customer_analysis.shape}")
print(f"Employee Performance: {employee_performance.shape}")
print(f"RPT by Service: {rpt_by_service.shape}")

✓ All cleaned datasets loaded!

Dataset Shapes:
Daily Revenue: (1276, 12)
Sale Ticket Detail: (3820, 31)
Customer Analysis: (1012, 6)
Employee Performance: (43, 4)
RPT by Service: (128, 6)


## 2. Time-Based Features

Extract comprehensive temporal features from dates for seasonality analysis.

In [17]:
# Create a comprehensive feature dataframe starting with daily revenue
features_df = daily_revenue[['Date', 'Revenue']].copy()

# Basic time features (already exist from EDA, but we'll ensure completeness)
features_df['Year'] = features_df['Date'].dt.year
features_df['Month'] = features_df['Date'].dt.month
features_df['Day'] = features_df['Date'].dt.day
features_df['DayOfWeek'] = features_df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
features_df['Quarter'] = features_df['Date'].dt.quarter
features_df['WeekOfYear'] = features_df['Date'].dt.isocalendar().week

# Binary flags
features_df['IsWeekend'] = (features_df['DayOfWeek'] >= 5).astype(int)
features_df['IsMonthStart'] = (features_df['Day'] <= 3).astype(int)  # First 3 days
features_df['IsMonthEnd'] = (features_df['Day'] >= features_df['Date'].dt.days_in_month - 2).astype(int)  # Last 3 days

# Cyclical encoding for circular features (better for ML models)
features_df['DayOfWeek_sin'] = np.sin(2 * np.pi * features_df['DayOfWeek'] / 7)
features_df['DayOfWeek_cos'] = np.cos(2 * np.pi * features_df['DayOfWeek'] / 7)
features_df['Month_sin'] = np.sin(2 * np.pi * features_df['Month'] / 12)
features_df['Month_cos'] = np.cos(2 * np.pi * features_df['Month'] / 12)
features_df['Day_sin'] = np.sin(2 * np.pi * features_df['Day'] / 31)
features_df['Day_cos'] = np.cos(2 * np.pi * features_df['Day'] / 31)

print("✓ Time-based features created!")
print(f"\nFeatures shape: {features_df.shape}")
print(f"\nSample:")
print(features_df.head())

✓ Time-based features created!

Features shape: (1276, 17)

Sample:
        Date  Revenue  Year  Month  Day  DayOfWeek  Quarter  WeekOfYear  \
0 2021-01-06    760.0  2021      1    6          2        1           1   
1 2021-10-07   1160.0  2021     10    7          3        4          40   
2 2021-10-09   5448.0  2021     10    9          5        4          40   
3 2021-10-10   1360.0  2021     10   10          6        4          40   
4 2021-10-11    760.0  2021     10   11          0        4          41   

   IsWeekend  IsMonthStart  IsMonthEnd  DayOfWeek_sin  DayOfWeek_cos  \
0          0             0           0       0.974928      -0.222521   
1          0             0           0       0.433884      -0.900969   
2          1             0           0      -0.974928      -0.222521   
3          1             0           0      -0.781831       0.623490   
4          0             0           0       0.000000       1.000000   

   Month_sin  Month_cos   Day_sin   Day_cos  
0 

## 3. Lag Features (Historical Patterns)

Create lagged revenue features to capture historical patterns and trends.

In [18]:
# Lag features - past revenue values
features_df['Revenue_Lag_7'] = features_df['Revenue'].shift(7)   # Same day last week
features_df['Revenue_Lag_14'] = features_df['Revenue'].shift(14) # Two weeks ago
features_df['Revenue_Lag_30'] = features_df['Revenue'].shift(30) # Same day last month

# Rolling statistics
features_df['Revenue_RollingMean_7'] = features_df['Revenue'].rolling(window=7, min_periods=1).mean()
features_df['Revenue_RollingMean_14'] = features_df['Revenue'].rolling(window=14, min_periods=1).mean()
features_df['Revenue_RollingMean_30'] = features_df['Revenue'].rolling(window=30, min_periods=1).mean()

features_df['Revenue_RollingStd_7'] = features_df['Revenue'].rolling(window=7, min_periods=1).std()
features_df['Revenue_RollingStd_14'] = features_df['Revenue'].rolling(window=14, min_periods=1).std()
features_df['Revenue_RollingStd_30'] = features_df['Revenue'].rolling(window=30, min_periods=1).std()

# Rolling min/max (for range)
features_df['Revenue_RollingMin_7'] = features_df['Revenue'].rolling(window=7, min_periods=1).min()
features_df['Revenue_RollingMax_7'] = features_df['Revenue'].rolling(window=7, min_periods=1).max()
features_df['Revenue_RollingMin_30'] = features_df['Revenue'].rolling(window=30, min_periods=1).min()
features_df['Revenue_RollingMax_30'] = features_df['Revenue'].rolling(window=30, min_periods=1).max()

# Exponential moving average (gives more weight to recent values)
features_df['Revenue_EMA_7'] = features_df['Revenue'].ewm(span=7, adjust=False).mean()
features_df['Revenue_EMA_30'] = features_df['Revenue'].ewm(span=30, adjust=False).mean()

print("✓ Lag and rolling features created!")
print(f"\nCurrent features: {features_df.shape[1]} columns")
print(f"\nSample lag features:")
print(features_df[['Date', 'Revenue', 'Revenue_Lag_7', 'Revenue_RollingMean_7', 'Revenue_RollingStd_7']].head(10))

✓ Lag and rolling features created!

Current features: 32 columns

Sample lag features:
        Date  Revenue  Revenue_Lag_7  Revenue_RollingMean_7  \
0 2021-01-06    760.0            NaN             760.000000   
1 2021-10-07   1160.0            NaN             960.000000   
2 2021-10-09   5448.0            NaN            2456.000000   
3 2021-10-10   1360.0            NaN            2182.000000   
4 2021-10-11    760.0            NaN            1897.600000   
5 2021-10-12  14488.0            NaN            3996.000000   
6 2021-10-14   4780.0            NaN            4108.000000   
7 2021-10-15   4800.0          760.0            4685.142857   
8 2021-10-16    380.0         1160.0            4573.714286   
9 2021-10-17   4412.0         5448.0            4425.714286   

   Revenue_RollingStd_7  
0                   NaN  
1            282.842712  
2           2598.855133  
3           2191.575385  
4           2001.666506  
5           5442.888057  
6           4977.482630  
7         

## 4. Malaysian Holiday Flags

Add binary flags for Malaysian public holidays and surrounding periods.

In [19]:
# Define Malaysian public holidays for the date range (2021-2025)
malaysian_holidays = {
    # 2021
    '2021-01-01': 'New Year',
    '2021-02-12': 'CNY Day 1',
    '2021-02-13': 'CNY Day 2',
    '2021-05-13': 'Hari Raya Aidilfitri',
    '2021-05-14': 'Hari Raya Aidilfitri Day 2',
    '2021-07-20': 'Hari Raya Aidiladha',
    '2021-08-10': 'Awal Muharram',
    '2021-08-31': 'Merdeka Day',
    '2021-09-16': 'Malaysia Day',
    '2021-10-19': 'Maulidur Rasul',
    '2021-11-04': 'Deepavali',
    '2021-12-25': 'Christmas',
    
    # 2022
    '2022-01-01': 'New Year',
    '2022-02-01': 'CNY Day 1',
    '2022-02-02': 'CNY Day 2',
    '2022-05-03': 'Hari Raya Aidilfitri',
    '2022-05-04': 'Hari Raya Aidilfitri Day 2',
    '2022-07-10': 'Hari Raya Aidiladha',
    '2022-07-30': 'Awal Muharram',
    '2022-08-31': 'Merdeka Day',
    '2022-09-16': 'Malaysia Day',
    '2022-10-09': 'Maulidur Rasul',
    '2022-10-24': 'Deepavali',
    '2022-12-25': 'Christmas',
    
    # 2023
    '2023-01-01': 'New Year',
    '2023-01-22': 'CNY Day 1',
    '2023-01-23': 'CNY Day 2',
    '2023-04-22': 'Hari Raya Aidilfitri',
    '2023-04-23': 'Hari Raya Aidilfitri Day 2',
    '2023-06-29': 'Hari Raya Aidiladha',
    '2023-07-19': 'Awal Muharram',
    '2023-08-31': 'Merdeka Day',
    '2023-09-16': 'Malaysia Day',
    '2023-09-28': 'Maulidur Rasul',
    '2023-11-12': 'Deepavali',
    '2023-12-25': 'Christmas',
    
    # 2024
    '2024-01-01': 'New Year',
    '2024-02-10': 'CNY Day 1',
    '2024-02-11': 'CNY Day 2',
    '2024-04-11': 'Hari Raya Aidilfitri',
    '2024-04-12': 'Hari Raya Aidilfitri Day 2',
    '2024-06-17': 'Hari Raya Aidiladha',
    '2024-07-07': 'Awal Muharram',
    '2024-08-31': 'Merdeka Day',
    '2024-09-16': 'Malaysia Day',
    '2024-09-16': 'Maulidur Rasul',
    '2024-10-31': 'Deepavali',
    '2024-12-25': 'Christmas',
    
    # 2025
    '2025-01-01': 'New Year',
    '2025-01-29': 'CNY Day 1',
    '2025-01-30': 'CNY Day 2',
    '2025-03-31': 'Hari Raya Aidilfitri',
    '2025-04-01': 'Hari Raya Aidilfitri Day 2',
    '2025-06-07': 'Hari Raya Aidiladha',
    '2025-06-27': 'Awal Muharram',
    '2025-08-31': 'Merdeka Day',
    '2025-09-05': 'Maulidur Rasul',
    '2025-09-16': 'Malaysia Day',
    '2025-10-20': 'Deepavali',
    '2025-12-25': 'Christmas',
}

# Convert to datetime
holiday_dates = pd.to_datetime(list(malaysian_holidays.keys()))

# Create holiday flag
features_df['IsHoliday'] = features_df['Date'].isin(holiday_dates).astype(int)

# Pre-holiday period (7 days before)
def is_pre_holiday(date, holidays, days_before=7):
    for holiday in holidays:
        if timedelta(days=0) < (holiday - date) <= timedelta(days=days_before):
            return 1
    return 0

# Post-holiday period (7 days after)
def is_post_holiday(date, holidays, days_after=7):
    for holiday in holidays:
        if timedelta(days=0) < (date - holiday) <= timedelta(days=days_after):
            return 1
    return 0

features_df['IsPreHoliday'] = features_df['Date'].apply(lambda x: is_pre_holiday(x, holiday_dates))
features_df['IsPostHoliday'] = features_df['Date'].apply(lambda x: is_post_holiday(x, holiday_dates))

# Major holidays (CNY, Hari Raya, Christmas, Deepavali)
major_holidays = ['CNY', 'Hari Raya', 'Christmas', 'Deepavali']
major_holiday_dates = [date for date, name in malaysian_holidays.items() if any(mh in name for mh in major_holidays)]
major_holiday_dates = pd.to_datetime(major_holiday_dates)
features_df['IsMajorHoliday'] = features_df['Date'].isin(major_holiday_dates).astype(int)

print("✓ Holiday features created!")
print(f"\nTotal holidays flagged: {features_df['IsHoliday'].sum()}")
print(f"Major holidays flagged: {features_df['IsMajorHoliday'].sum()}")
print(f"Pre-holiday periods: {features_df['IsPreHoliday'].sum()}")
print(f"Post-holiday periods: {features_df['IsPostHoliday'].sum()}")
print(f"\nSample holidays:")
print(features_df[features_df['IsHoliday'] == 1][['Date', 'Revenue', 'IsHoliday', 'IsMajorHoliday']].head(10))

✓ Holiday features created!

Total holidays flagged: 33
Major holidays flagged: 14
Pre-holiday periods: 250
Post-holiday periods: 235

Sample holidays:
          Date  Revenue  IsHoliday  IsMajorHoliday
10  2021-10-19  11816.0          1               0
21  2021-11-04    760.0          1               1
69  2021-12-25    376.0          1               1
76  2022-01-01    578.0          1               0
260 2022-07-10   7744.0          1               1
280 2022-07-30   7672.0          1               0
312 2022-08-31   5526.0          1               0
327 2022-09-16   2546.0          1               0
350 2022-10-09   6508.0          1               0
365 2022-10-24   2598.0          1               1


## 5. Business Context Features

Add features related to customer behavior, employee performance, and service mix.

In [20]:
# Aggregate daily transaction data from sale_ticket_clean
daily_transactions = sale_ticket_clean.groupby(sale_ticket_clean['Date'].dt.date).agg({
    'Reference No.': 'nunique',  # Number of transactions
    'Customer': 'nunique',        # Number of unique customers
    'Employee': 'nunique',        # Number of unique employees working
    'Total (MYR)': ['sum', 'mean', 'std'],  # Revenue stats
    'Discount (MYR)': 'sum',      # Total discounts given
    'FOC (MYR)': 'sum',           # Free of charge amount
}).reset_index()

# Flatten column names
daily_transactions.columns = ['Date', 'TransactionCount', 'UniqueCustomers', 'UniqueEmployees',
                               'TotalRevenue', 'AvgTransactionValue', 'StdTransactionValue',
                               'TotalDiscount', 'TotalFOC']

# Convert Date to datetime
daily_transactions['Date'] = pd.to_datetime(daily_transactions['Date'])

# Merge with features_df
features_df = features_df.merge(daily_transactions, on='Date', how='left')

# Fill NaN with 0 (days with no transactions)
transaction_cols = ['TransactionCount', 'UniqueCustomers', 'UniqueEmployees',
                    'TotalRevenue', 'AvgTransactionValue', 'StdTransactionValue',
                    'TotalDiscount', 'TotalFOC']
features_df[transaction_cols] = features_df[transaction_cols].fillna(0)

# Calculate derived features
features_df['AvgRevenuePerCustomer'] = features_df['TotalRevenue'] / (features_df['UniqueCustomers'] + 1)  # +1 to avoid division by zero
features_df['AvgRevenuePerEmployee'] = features_df['TotalRevenue'] / (features_df['UniqueEmployees'] + 1)
features_df['DiscountRate'] = features_df['TotalDiscount'] / (features_df['TotalRevenue'] + features_df['TotalDiscount'] + 1)
features_df['FOCRate'] = features_df['TotalFOC'] / (features_df['TotalRevenue'] + 1)

print("✓ Business context features created!")
print(f"\nCurrent features: {features_df.shape[1]} columns")
print(f"\nSample business features:")
print(features_df[['Date', 'TransactionCount', 'UniqueCustomers', 'AvgTransactionValue', 'DiscountRate']].head(10))

✓ Business context features created!

Current features: 48 columns

Sample business features:
        Date  TransactionCount  UniqueCustomers  AvgTransactionValue  \
0 2021-01-06               2.0              2.0           190.000000   
1 2021-10-07               1.0              1.0           240.000000   
2 2021-10-09               3.0              3.0           561.333333   
3 2021-10-10               2.0              2.0            95.000000   
4 2021-10-11               1.0              1.0           190.000000   
5 2021-10-12               4.0              3.0           595.000000   
6 2021-10-14               2.0              2.0           715.000000   
7 2021-10-15               1.0              1.0          2400.000000   
8 2021-10-16               1.0              1.0           190.000000   
9 2021-10-17               3.0              1.0           382.666667   

   DiscountRate  
0      0.499343  
1      0.498960  
2      0.667062  
3      0.498688  
4      0.498688  
5    

## 6. Service Mix Features

Analyze the diversity and composition of services provided each day.

In [21]:
# Service type distribution by day
service_daily = sale_ticket_clean[sale_ticket_clean['Type'] == 'S'].copy()
service_daily['Date_Only'] = service_daily['Date'].dt.date

# Count services by type per day
service_type_counts = service_daily.groupby(['Date_Only', 'Type']).size().unstack(fill_value=0)
service_type_counts = service_type_counts.reset_index()
service_type_counts['Date_Only'] = pd.to_datetime(service_type_counts['Date_Only'])

# Service diversity: Shannon entropy
from scipy.stats import entropy

def calculate_service_diversity(date):
    services = service_daily[service_daily['Date_Only'] == date]['Item Name']
    if len(services) == 0:
        return 0
    service_counts = services.value_counts()
    return entropy(service_counts, base=2)  # Shannon entropy

# Calculate for each date
unique_dates = service_daily['Date_Only'].unique()
diversity_data = []
for date in unique_dates:
    diversity = calculate_service_diversity(date)
    unique_services = service_daily[service_daily['Date_Only'] == date]['Item Name'].nunique()
    diversity_data.append({
        'Date': pd.to_datetime(date),
        'ServiceDiversity': diversity,
        'UniqueServices': unique_services
    })

diversity_df = pd.DataFrame(diversity_data)

# Merge with features
features_df = features_df.merge(diversity_df, on='Date', how='left')
features_df['ServiceDiversity'] = features_df['ServiceDiversity'].fillna(0)
features_df['UniqueServices'] = features_df['UniqueServices'].fillna(0)

print("✓ Service mix features created!")
print(f"\nCurrent features: {features_df.shape[1]} columns")
print(f"\nSample service features:")
print(features_df[['Date', 'Revenue', 'UniqueServices', 'ServiceDiversity']].head(10))

✓ Service mix features created!

Current features: 50 columns

Sample service features:
        Date  Revenue  UniqueServices  ServiceDiversity
0 2021-01-06    760.0             1.0          0.000000
1 2021-10-07   1160.0             1.0          0.000000
2 2021-10-09   5448.0             3.0          1.584963
3 2021-10-10   1360.0             1.0          0.000000
4 2021-10-11    760.0             1.0          0.000000
5 2021-10-12  14488.0             3.0          1.500000
6 2021-10-14   4780.0             2.0          1.000000
7 2021-10-15   4800.0             1.0          0.000000
8 2021-10-16    380.0             1.0          0.000000
9 2021-10-17   4412.0             1.0          0.000000


## 7. Interaction Features

Create features that capture interactions between different variables.

In [22]:
# Interaction features
features_df['Weekend_x_Month'] = features_df['IsWeekend'] * features_df['Month']
features_df['Holiday_x_Weekend'] = features_df['IsHoliday'] * features_df['IsWeekend']
features_df['TransactionCount_x_AvgValue'] = features_df['TransactionCount'] * features_df['AvgTransactionValue']
features_df['UniqueCustomers_x_UniqueEmployees'] = features_df['UniqueCustomers'] * features_df['UniqueEmployees']
features_df['ServiceDiversity_x_TransactionCount'] = features_df['ServiceDiversity'] * features_df['TransactionCount']
features_df['DiscountRate_x_IsWeekend'] = features_df['DiscountRate'] * features_df['IsWeekend']
features_df['MonthEnd_x_Revenue'] = features_df['IsMonthEnd'] * features_df['Revenue']

print("✓ Interaction features created!")
print(f"\nTotal features: {features_df.shape[1]} columns")

✓ Interaction features created!

Total features: 57 columns


## 8. Feature Summary and Correlation Analysis

In [23]:
# List all features
print("="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

feature_categories = {
    'Time-Based': ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'WeekOfYear', 'IsWeekend', 
                   'IsMonthStart', 'IsMonthEnd', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 
                   'Month_cos', 'Day_sin', 'Day_cos'],
    'Lag Features': ['Revenue_Lag_7', 'Revenue_Lag_14', 'Revenue_Lag_30', 'Revenue_RollingMean_7',
                     'Revenue_RollingMean_14', 'Revenue_RollingMean_30', 'Revenue_RollingStd_7',
                     'Revenue_RollingStd_14', 'Revenue_RollingStd_30', 'Revenue_RollingMin_7',
                     'Revenue_RollingMax_7', 'Revenue_RollingMin_30', 'Revenue_RollingMax_30',
                     'Revenue_EMA_7', 'Revenue_EMA_30'],
    'Holiday Flags': ['IsHoliday', 'IsPreHoliday', 'IsPostHoliday', 'IsMajorHoliday'],
    'Business Context': ['TransactionCount', 'UniqueCustomers', 'UniqueEmployees', 'TotalRevenue',
                         'AvgTransactionValue', 'StdTransactionValue', 'TotalDiscount', 'TotalFOC',
                         'AvgRevenuePerCustomer', 'AvgRevenuePerEmployee', 'DiscountRate', 'FOCRate'],
    'Service Mix': ['ServiceDiversity', 'UniqueServices'],
    'Interactions': ['Weekend_x_Month', 'Holiday_x_Weekend', 'TransactionCount_x_AvgValue',
                     'UniqueCustomers_x_UniqueEmployees', 'ServiceDiversity_x_TransactionCount',
                     'DiscountRate_x_IsWeekend', 'MonthEnd_x_Revenue']
}

for category, features in feature_categories.items():
    existing_features = [f for f in features if f in features_df.columns]
    print(f"\n{category}: {len(existing_features)} features")
    print(f"  {', '.join(existing_features)}")

print(f"\n{'='*80}")
print(f"TOTAL FEATURES: {features_df.shape[1]}")
print(f"TOTAL RECORDS: {features_df.shape[0]}")
print(f"{'='*80}")

FEATURE ENGINEERING SUMMARY

Time-Based: 15 features
  Year, Month, Day, DayOfWeek, Quarter, WeekOfYear, IsWeekend, IsMonthStart, IsMonthEnd, DayOfWeek_sin, DayOfWeek_cos, Month_sin, Month_cos, Day_sin, Day_cos

Lag Features: 15 features
  Revenue_Lag_7, Revenue_Lag_14, Revenue_Lag_30, Revenue_RollingMean_7, Revenue_RollingMean_14, Revenue_RollingMean_30, Revenue_RollingStd_7, Revenue_RollingStd_14, Revenue_RollingStd_30, Revenue_RollingMin_7, Revenue_RollingMax_7, Revenue_RollingMin_30, Revenue_RollingMax_30, Revenue_EMA_7, Revenue_EMA_30

Holiday Flags: 4 features
  IsHoliday, IsPreHoliday, IsPostHoliday, IsMajorHoliday

Business Context: 12 features
  TransactionCount, UniqueCustomers, UniqueEmployees, TotalRevenue, AvgTransactionValue, StdTransactionValue, TotalDiscount, TotalFOC, AvgRevenuePerCustomer, AvgRevenuePerEmployee, DiscountRate, FOCRate

Service Mix: 2 features
  ServiceDiversity, UniqueServices

Interactions: 7 features
  Weekend_x_Month, Holiday_x_Weekend, TransactionC

In [24]:
# Check for missing values
missing = features_df.isnull().sum()
missing_pct = (missing / len(features_df)) * 100
missing_df = pd.DataFrame({
    'Feature': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("\nMissing Values Summary:")
    print(missing_df)
else:
    print("\n✓ No missing values in feature set!")


Missing Values Summary:
                  Feature  Missing_Count  Missing_Percentage
19         Revenue_Lag_30             30            2.351097
18         Revenue_Lag_14             14            1.097179
17          Revenue_Lag_7              7            0.548589
23   Revenue_RollingStd_7              1            0.078370
24  Revenue_RollingStd_14              1            0.078370
25  Revenue_RollingStd_30              1            0.078370


## 9. Feature Visualization

## 10. Save Engineered Features

In [25]:
# Create features directory
import os
os.makedirs('../data/features', exist_ok=True)

# Save complete feature set
features_df.to_csv('../data/features/features_complete.csv', index=False)

# Save feature names for reference
feature_names = {
    'all_features': features_df.columns.tolist(),
    'categories': feature_categories
}

import json
with open('../data/features/feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)
print(f"\n✓ {features_df.shape[1]} features created for {features_df.shape[0]} records")
print(f"✓ Saved to: ../data/features/features_complete.csv")
print(f"✓ Feature metadata saved to: ../data/features/feature_names.json")
print(f"\nFeature file size: {os.path.getsize('../data/features/features_complete.csv') / 1024 / 1024:.2f} MB")
print("\n" + "="*80)


FEATURE ENGINEERING COMPLETE

✓ 57 features created for 1276 records
✓ Saved to: ../data/features/features_complete.csv
✓ Feature metadata saved to: ../data/features/feature_names.json

Feature file size: 0.56 MB



## 11. Feature Statistics Summary

In [26]:
# Generate comprehensive statistics
print("Feature Statistics Summary:\n")
print(features_df.describe().T)

# Check data types
print("\n" + "="*80)
print("Data Types:")
print(features_df.dtypes.value_counts())

print("\n✓ Part 1.3: Feature Engineering COMPLETE")

Feature Statistics Summary:

                                      count                           mean  \
Date                                   1276  2023-09-10 14:09:46.833855744   
Revenue                              1276.0                    2978.244436   
Year                                 1276.0                    2023.181034   
Month                                1276.0                       6.663793   
Day                                  1276.0                      15.682602   
DayOfWeek                            1276.0                       3.106583   
Quarter                              1276.0                       2.554859   
WeekOfYear                           1276.0                      27.124608   
IsWeekend                            1276.0                       0.307994   
IsMonthStart                         1276.0                       0.096395   
IsMonthEnd                           1276.0                       0.101881   
DayOfWeek_sin                      